# Retail Sales Exploratory Data Analysis (EDA)
**Internship Track:** Data Analytics  
**Project:** OASIS INFOBYTE — Level 1, Task 1

## Objective
Explore retail sales data to identify sales trends, customer patterns, and product-category performance.

## Dataset
[Kaggle Retail Sales Dataset](https://www.kaggle.com/datasets/mohammadtalib786/retail-sales-dataset)

**Important limitation:** This dataset has product categories but no individual product names, so individual best-selling products cannot be identified.

In [ ]:
import os, glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
file_path = "/kaggle/input/datasets/mohammadtalib786/retail-sales-dataset/retail_sales_dataset.csv"
if not os.path.exists(file_path):
    matches = glob.glob("/kaggle/input/**/retail_sales_dataset.csv", recursive=True)
    if matches:
        file_path = matches[0]
    else:
        raise FileNotFoundError("Add the Retail Sales Dataset to this Kaggle notebook, then run this cell again.")

df = pd.read_csv(file_path)
print("Shape:", df.shape)
display(df.head())

## 1. Dataset inspection
Check data types, missing values, duplicate rows, and descriptive statistics.

In [ ]:
print("Columns:", df.columns.tolist())
df.info()
print("\nMissing values by column:")
display(df.isnull().sum().to_frame("Missing values"))
print("\nDuplicate rows:", df.duplicated().sum())
display(df.describe(include="all").T)

### Observations
Review the outputs above and record any missing values or duplicates. Inspect any issues before changing or removing data.

## 2. Prepare dates and numeric columns

In [ ]:
required = ["Date", "Gender", "Age", "Product Category", "Quantity", "Price per Unit", "Total Amount"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
for c in ["Age", "Quantity", "Price per Unit", "Total Amount"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["Month"] = df["Date"].dt.to_period("M")
df["Quarter"] = df["Date"].dt.to_period("Q")
print("Date range:", df["Date"].min(), "to", df["Date"].max())
print("Invalid dates:", df["Date"].isna().sum())

## 3. Descriptive statistics
Mean, median, mode, and standard deviation for numeric columns.

In [ ]:
numeric = df.select_dtypes(include="number").columns
stats = pd.DataFrame({
    "Mean": df[numeric].mean(),
    "Median": df[numeric].median(),
    "Mode": df[numeric].mode().iloc[0],
    "Standard Deviation": df[numeric].std()
})
display(stats.round(2))

## 4. Monthly sales

In [ ]:
monthly = df.dropna(subset=["Date"]).groupby("Month")["Total Amount"].sum().sort_index()
display(monthly.to_frame("Total Sales"))
plt.figure(figsize=(12,5))
plt.plot(monthly.index.astype(str), monthly.values, marker="o")
plt.title("Monthly Retail Sales"); plt.xlabel("Month"); plt.ylabel("Total Sales")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()
if not monthly.empty:
    print("Highest month:", monthly.idxmax(), f"({monthly.max():,.0f})")
    print("Lowest month:", monthly.idxmin(), f"({monthly.min():,.0f})")

### Monthly observations
The earlier analysis found May 2023 had the highest monthly sales (53,150). January 2024 may include only January 1, so treat it as an incomplete month. Use complete-month trends to inform inventory and promotions.

## 5. Quarterly sales

In [ ]:
quarterly = df.dropna(subset=["Date"]).groupby("Quarter")["Total Amount"].sum().sort_index()
display(quarterly.to_frame("Total Sales"))
plt.figure(figsize=(10,5))
plt.plot(quarterly.index.astype(str), quarterly.values, marker="o")
plt.title("Quarterly Retail Sales"); plt.xlabel("Quarter"); plt.ylabel("Total Sales")
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

### Quarterly observations
The earlier analysis found Q4 2023 had the highest complete-quarter sales (126,190). Q1 2024 is incomplete in this dataset and should not be compared directly with complete quarters. Use complete periods for seasonal planning.

## 6. Customer gender

In [ ]:
gender_counts = df["Gender"].value_counts(dropna=False)
display(gender_counts.to_frame("Transaction Count"))
plt.figure(figsize=(7,5))
sns.barplot(x=gender_counts.index.astype(str), y=gender_counts.values)
plt.title("Transactions by Gender"); plt.xlabel("Gender"); plt.ylabel("Transactions")
plt.tight_layout(); plt.show()

### Gender observations
The earlier analysis found 510 female transactions (51%) and 490 male transactions (49%), a nearly balanced distribution. Test campaigns across groups instead of relying on this small difference.

## 7. Customer age groups

In [ ]:
bins = [17,25,35,45,55,65]
labels = ["18-25","26-35","36-45","46-55","56-64"]
df["Age Group"] = pd.cut(df["Age"], bins=bins, labels=labels, include_lowest=True)
age_counts = df["Age Group"].value_counts().sort_index()
display(age_counts.to_frame("Transaction Count"))
plt.figure(figsize=(8,5))
sns.barplot(x=age_counts.index.astype(str), y=age_counts.values)
plt.title("Transactions by Age Group"); plt.xlabel("Age Group"); plt.ylabel("Transactions")
plt.tight_layout(); plt.show()

### Age observations
The earlier analysis found ages 46–55 had the highest transaction count (229; 22.9%), while ages 18–25 had the lowest (169; 16.9%). Consider testing offers for different age groups.

## 8. Revenue by product category

In [ ]:
category_revenue = df.groupby("Product Category")["Total Amount"].sum().sort_values(ascending=False)
display(category_revenue.to_frame("Total Revenue"))
plt.figure(figsize=(8,5))
sns.barplot(x=category_revenue.index, y=category_revenue.values)
plt.title("Total Revenue by Product Category"); plt.xlabel("Category"); plt.ylabel("Total Revenue")
plt.tight_layout(); plt.show()

### Category observations
The earlier analysis found Electronics generated the highest total revenue (156,905), followed by Clothing (155,580) and Beauty (143,515). Revenue is not profit because costs are not included.

## 9. Top 10 high-quantity transactions — dataset limitation
There are no individual product names in this dataset. This table shows the ten transactions with the largest quantities, **not** the ten best-selling individual products.

In [ ]:
top_transactions = df.nlargest(10, "Quantity")
show_cols = [c for c in ["Transaction ID","Date","Product Category","Quantity","Price per Unit","Total Amount"] if c in df.columns]
display(top_transactions[show_cols])

To rank best-selling individual products, use a dataset containing product names or product IDs and quantities sold.

## 10. Correlation heatmap

In [ ]:
corr_cols = ["Age","Quantity","Price per Unit","Total Amount"]
corr = df[corr_cols].corr()
plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Correlation Heatmap"); plt.tight_layout(); plt.show()
display(corr.round(2))

### Correlation observations
Earlier results showed Price per Unit and Total Amount had a strong positive correlation (about 0.85), while Quantity and Total Amount had a moderate positive correlation (about 0.37). Correlation indicates association, not causation.

## 11. Extra visualization: average transaction value by category

In [ ]:
avg_value = df.groupby("Product Category")["Total Amount"].mean().sort_values(ascending=False)
display(avg_value.to_frame("Average Transaction Value").round(2))
plt.figure(figsize=(8,5))
sns.barplot(x=avg_value.index, y=avg_value.values)
plt.title("Average Transaction Value by Category"); plt.xlabel("Category"); plt.ylabel("Average Transaction Value")
plt.tight_layout(); plt.show()

### Extra chart observations
Earlier results showed Beauty had the highest average transaction value (467.48), followed by Electronics (458.79) and Clothing (443.25). Consider testing bundles or cross-selling in Clothing and measuring the results.

## 12. Conclusion and actionable recommendations

1. **Inventory planning:** Use complete monthly and quarterly periods to identify demand patterns.
2. **Category monitoring:** Track Electronics and Clothing revenue and investigate opportunities in Beauty.
3. **Customer marketing:** Test campaigns across age groups and genders and compare results.
4. **Cross-selling:** Test bundles in categories with lower average transaction value.
5. **Regular review:** Monitor revenue, transaction counts, and average transaction value over time.

### Limitations
- The dataset contains categories but no individual product names, so best-selling individual products cannot be ranked.
- January 2024 may include only January 1 and is incomplete.
- Revenue is not profit; cost and margin data are unavailable.

These exploratory findings should be checked against additional periods and business context before business decisions are made.